# EONET API → PostgreSQL
## Notebook z Projektem bazy danych oraz importem bazy danych


<div style="text-align: center;">
    <img src="Onet.svg" width="900"  alt="Diagram bazy danych" />
</div>

### Encje (Tabele)
W schemacie występują następujące tabele:

* `event` - przechowuje podstawowe informacje o zdarzeniach (np. zjawiskach pobranych z API, z obciętym ID).
* `event_location` - przechowuje dane adresowe (kraj, miasto) konkretnego zdarzenia.
* `geometry` - przechowuje dokładne współrzędne geograficzne, daty pomiaru oraz dodatkowe parametry (magnitude, unit).
* `geometry_type` - tabela słownikowa przechowująca typy obiektów geometrycznych.
* `category` - tabela słownikowa przechowująca kategorie zdarzeń.
* `source` - tabela słownikowa przechowująca źródła danych dla zdarzeń.
* `event_category` - tabela asocjacyjna łącząca zdarzenia z odpowiednimi kategoriami.
* `event_source` - tabela asocjacyjna łącząca zdarzenia z odpowiednimi źródłami.
* `event_geometry` - tabela asocjacyjna łącząca zdarzenia z przypisanymi im obiektami geometrycznymi.
* `import_log` - rejestr zdarzeń przechowujący informacje o procesach importu danych z API (logi).



### Lista  zależności:

| Zależność | Znaczenie |
| :--- | :--- |
| `event_location.event_id` &rarr; `event.id` | lokalizacja jest ściśle powiązana z konkretnym zdarzeniem |
| `geometry.type_id` &rarr; `geometry_type.id` | każdy rekord geometrii musi mieć określony typ |
| `event` &rarr; `category` | zdarzenie jest przypisane do jedenego  lub wielu  kategorii |
| `event` &rarr; `source` | informacja o zdarzeniu pochodzi z jednego  lub  wielu źródeł |
| `event` &rarr; `geometry` | zdarzenie ma przypisane jedne  lub wiele dany współrzędnych |

## Skrypt tworzący baze danych

```sql

DROP TABLE IF EXISTS import_log;
DROP TABLE IF EXISTS event_category;
DROP TABLE IF EXISTS event_source;
DROP TABLE IF EXISTS event_geometry;
DROP TABLE IF EXISTS source;
DROP TABLE IF EXISTS category;
DROP TABLE IF EXISTS geometry;
DROP TABLE IF EXISTS geometry_type;
DROP TABLE IF EXISTS event_location;
DROP TABLE IF EXISTS event;

CREATE TABLE event (
	id INTEGER PRIMARY KEY, -- Przychodzi z API jako EONET_x zapisujemy tylko x
	title  VARCHAR(255) NOT NULL, -- Do przecinka z tego co przydchodzi w polu title
	description TEXT,
	start_date TIMESTAMP NOT NULL,
	end_date TIMESTAMP
);

CREATE TABLE event_location (
    id SERIAL PRIMARY KEY,
    event_id INTEGER NOT NULL, 
    country_code CHAR(2),
    country VARCHAR(100),
    state VARCHAR(100),
    city VARCHAR(100),

   
    CONSTRAINT fk_event_location_event
        FOREIGN KEY (event_id) 
        REFERENCES event(id)
		ON UPDATE CASCADE 
        ON DELETE CASCADE
);

CREATE TABLE geometry_type (
	id SERIAL PRIMARY KEY,
	type VARCHAR(100) NOT NULL
);

CREATE TABLE geometry (
	id SERIAL PRIMARY KEY, -- Własny z sql
	latitude FLOAT NOT NULL CHECK (90 >= latitude AND latitude >= -90), --latitude i longitude są w takich zakresach
	longitude FLOAT NOT NULL CHECK (180 >= longitude AND longitude >= -180),
	type_id INTEGER NOT NULL,
	magnitude FLOAT,
	unit VARCHAR(50),
	measure_date TIMESTAMP NOT NULL,

	CONSTRAINT fk_geometry_type_geometry
		FOREIGN KEY (type_id)
		REFERENCES geometry_type(id)
		ON UPDATE CASCADE
		ON DELETE RESTRICT --nie chcemy usunąć typu punktu który jest zapisany w bazie
);

CREATE TABLE category (
	id SERIAL PRIMARY KEY,
	abbreviation VARCHAR(50) NOT NULL,
	title VARCHAR(255) NOT NULL,
	description TEXT
);

CREATE TABLE source (
	id SERIAL PRIMARY KEY,
	abbreviation VARCHAR(50) NOT NULL,
	title VARCHAR(255) NOT NULL,
	description TEXT
	
);

CREATE TABLE event_category (
	event_id INTEGER NOT NULL,
	category_id INTEGER NOT NULL,

	CONSTRAINT pk_event_category
		PRIMARY KEY (event_id, category_id),
		
	CONSTRAINT fk_event_category_event
		FOREIGN KEY (event_id)
		REFERENCES event(id)
		ON UPDATE CASCADE
		ON DELETE CASCADE,

	CONSTRAINT fk_event_category_category
		FOREIGN KEY (category_id)
		REFERENCES category(id)
		ON UPDATE CASCADE
		ON DELETE CASCADE
);

CREATE TABLE event_source (
	event_id INTEGER  NOT NULL,
	source_id INTEGER NOT  NULL,

	
	CONSTRAINT pk_event_source
		PRIMARY KEY (event_id, source_id),

	CONSTRAINT fk_event_source_event
		FOREIGN KEY (event_id)
		REFERENCES event(id)
		ON UPDATE CASCADE
		ON DELETE CASCADE,

	CONSTRAINT fk_event_source_source
		FOREIGN KEY (source_id)
		REFERENCES source(id)
		ON UPDATE CASCADE
		ON DELETE CASCADE
);

CREATE TABLE event_geometry (
	event_id INTEGER NOT NULL,
	geometry_id INTEGER NOT NULL,


	CONSTRAINT pk_event_geometry
		PRIMARY KEY (event_id, geometry_id),

	CONSTRAINT fk_event_geometry_event
		FOREIGN KEY (event_id)
		REFERENCES event(id)
		ON UPDATE CASCADE
		ON DELETE CASCADE,

	CONSTRAINT fk_event_geometry
		FOREIGN KEY (geometry_id)
		REFERENCES geometry(id)
		ON UPDATE CASCADE
		ON DELETE CASCADE

);

CREATE TABLE import_log (
    id SERIAL PRIMARY KEY,
    imported_at TIMESTAMP NOT NULL,
    apiname VARCHAR(255) NOT NULL,
    endpoint VARCHAR(255) NOT NULL,
    records_received INTEGER NOT NULL,
    records_saved INTEGER NOT NULL,
    status VARCHAR(100) NOT NULL,
    error_message TEXT NOT NULL
);
```

### Restrykcje nałożone na dane

* **event.id** - INTEGER pobierany API *EONET*
* **geometry.latitude**  - Szerokość geograficzna od -90 do 90
* **geometry.longtitude** - Długość geograficzna od -180 do 180
* **geometry.type_id** - ON DELETE RESTRICT - Nie usuwamy typu punktu którego rodzaj istnieje w tabeli geometry



## Import bibliotek

In [1]:
from __future__ import annotations
from typing import Any
from time import sleep


from sqlalchemy import URL, create_engine, text
from sqlalchemy.exc import SQLAlchemyError
import pandas as pd
import reverse_geocode as rg
import requests

## Tworzenie schematu bazy danych

##### Struktura pliku konfiguracyjnego (conf.txt):
DB_USER=moj_uzytkownik\
DB_PASSWORD=moje_super_tajne_haslo\
DB_HOST=localhost\
DB_PORT=5432\
DB_NAME=moja_baza_danych

In [2]:


config = {}
with open("conf.txt", "r", encoding="utf-8") as f:
    for linia in f:
        linia = linia.strip()
        if not linia or linia.startswith("#"):
            continue
        klucz, wartosc = linia.split("=", 1)
        config[klucz.strip()] = wartosc.strip()


db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=config.get("DB_USER"),
    password=config.get("DB_PASSWORD"),
    host=config.get("DB_HOST"),
    port=config.get("DB_PORT"),      
    database=config.get("DB_NAME"),
)


from scripts import schema

engine = create_engine(db_url)


In [3]:

try:
    conn = engine.connect()
    print("[INFO] POŁACZONO DO BAZY DANYCH")
except SQLAlchemyError as e:
    print(f"[ERROR] BŁĄD POŁĄCZENIA DO BAZY DANYCH {e}")

res = conn.execute(text(schema))
conn.commit()
print("[INFO] UTWORZONO BAZĘ DANYCH")
conn.close()

[INFO] POŁACZONO DO BAZY DANYCH
[INFO] UTWORZONO BAZĘ DANYCH


## Pobieranie danych z API


In [4]:
from  fetch_api import fetch_events, fetch_categories, fetch_source
DAYS = 100
LIMIT = 100
events = fetch_events(DAYS, LIMIT)
print(events[0:3])
sources = fetch_source()
print(sources[0:3])
categories = fetch_categories()
print(categories[0:3])

[{'id': 'EONET_20292', 'title': 'Flood in Türkiye 1103920', 'description': None, 'link': 'https://eonet.gsfc.nasa.gov/api/v3/events/EONET_20292', 'closed': '2026-06-29T00:00:00Z', 'categories': [{'id': 'floods', 'title': 'Floods'}], 'sources': [{'id': 'GDACS', 'url': 'https://www.gdacs.org/report.aspx?eventtype=FL&eventid=1103920'}], 'geometry': [{'magnitudeValue': None, 'magnitudeUnit': None, 'date': '2026-06-27T20:00:00Z', 'type': 'Point', 'coordinates': [36.4534072, 37.2401223]}]}, {'id': 'EONET_20484', 'title': 'Wildfire in Zambia 1028904', 'description': None, 'link': 'https://eonet.gsfc.nasa.gov/api/v3/events/EONET_20484', 'closed': '2026-06-13T00:00:00Z', 'categories': [{'id': 'wildfires', 'title': 'Wildfires'}], 'sources': [{'id': 'GDACS', 'url': 'https://www.gdacs.org/report.aspx?eventtype=WF&eventid=1028904'}], 'geometry': [{'magnitudeValue': 7491.0, 'magnitudeUnit': 'hectare', 'date': '2026-06-12T19:00:00Z', 'type': 'Point', 'coordinates': [25.443081972078488, -15.3465933037

## Zapisywanie danych do tabel

In [5]:
from import_danych import import_event, import_categories, import_sources

import_sources(sources, engine)
import_categories(categories, engine)
import_event(events, engine)



[INFO] Zaimportowano 33 źródeł
[INFO] Zaimportowano 13 kategorii
[INFO] Zapisano event 20308
[INFO] Import zakończony


## Weryfikacja zawartości bazy danych

## Skrypty do analizy  lub  raportów

####  Najczęstsze  zjawiska
```sql
SELECT 
    c.title AS category_name, 
    c.abbreviation AS abbreviation,
    COUNT(ec.event_id) AS amount_of_events
FROM category c
JOIN event_category ec ON c.id = ec.category_id
GROUP BY c.title, c.abbreviation
ORDER BY amount_of_events DESC
LIMIT 10;
```

In [6]:
from scripts import analyze_1 as most_popular

with engine.begin() as conn:
    res = pd.read_sql(most_popular, con=conn)
    display(res)

,category_name,abbreviation,amount_of_events
0,Wildfires,wildfires,88
1,Floods,floods,10
2,Severe Storms,severeStorms,2


#### Liczba zjawisk w poszczególnych krajach
```sql
SELECT 
    el.country AS country, 
    COUNT(e.id) AS amount_of_events
FROM event e
JOIN event_location el ON e.id = el.event_id
WHERE el.country IS NOT NULL
GROUP BY el.country
ORDER BY amount_of_events DESC
LIMIT 10;
```

In [7]:
from scripts import analyze_2 as by_country

with engine.begin() as conn:
    res = pd.read_sql(by_country, con=conn)
    display(res)

,country,amount_of_events
0,Australia,32
1,"Congo, The Democratic Republic of the",16
2,United States,15
3,Angola,15
4,Zambia,3
5,El Salvador,2
6,Canada,2
7,Russian Federation,2
8,Ghana,1
9,Algeria,1


#### Aktywne zjawiska
```sql
SELECT 
    e.title AS event_title,
    e.start_date AS start_date,
    s.title AS data_source
FROM event e
JOIN event_source es ON e.id = es.event_id
JOIN source s ON es.source_id = s.id
WHERE e.end_date IS NULL
ORDER BY e.start_date DESC
LIMIT 10;
```

In [8]:
from scripts import analyze_3 as active

with engine.begin() as conn:
    res = pd.read_sql(active, con=conn)
    display(res)

,event_title,start_date,data_source
0,Little Schloss RX Prescribed Fire,2026-06-10 14:51:00,Integrated Reporting of Wildfire Information (...
1,South Fork Wildfire,2026-06-09 20:08:00,Integrated Reporting of Wildfire Information (...
2,APQ Area 11 Scrub RX 0608 Prescribed Fire,2026-06-09 11:20:00,Integrated Reporting of Wildfire Information (...
3,South Mountain Wildfire,2026-06-09 00:15:00,Integrated Reporting of Wildfire Information (...
4,PUTAH Wildfire,2026-06-08 14:35:00,Integrated Reporting of Wildfire Information (...
5,Tropical Storm Cristina,2026-06-08 12:00:00,National Hurricane Center
6,Tropical Storm Boris,2026-06-08 06:00:00,National Hurricane Center
7,Tower Wildfire,2026-06-07 15:24:00,Integrated Reporting of Wildfire Information (...
8,LXR Lower Third RX 0609 Prescribed Fire,2026-06-07 14:01:00,Integrated Reporting of Wildfire Information (...
9,Raven Creek Wildfire,2026-06-06 20:44:00,Integrated Reporting of Wildfire Information (...


#### Zdarzenie z największą siła w  danej kategorii
```sql
SELECT DISTINCT
    c.title AS category_name,
    e.title AS event_title,
    g.magnitude AS max_magnitude
FROM category c
JOIN event_category ec ON c.id = ec.category_id
JOIN event e ON ec.event_id = e.id
JOIN event_geometry eg ON e.id = eg.event_id
JOIN geometry g ON eg.geometry_id = g.id
WHERE e.id = (
    SELECT ec_sub.event_id
    FROM event_category ec_sub
    JOIN event_geometry eg_sub ON ec_sub.event_id = eg_sub.event_id
    JOIN geometry g_sub ON eg_sub.geometry_id = g_sub.id
    WHERE ec_sub.category_id = c.id
    ORDER BY g_sub.magnitude DESC NULLS LAST
    LIMIT 1
)
AND g.magnitude = (
    SELECT MAX(g_sub2.magnitude) 
    FROM event_geometry eg_sub2 
    JOIN geometry g_sub2 ON eg_sub2.geometry_id = g_sub2.id 
    WHERE eg_sub2.event_id = e.id
); 
```

In [9]:
from scripts import analyze_4 as max_from_category

with engine.begin() as conn:
    res = pd.read_sql(max_from_category, con=conn)
    display(res)

,category_name,event_title,max_magnitude
0,Severe Storms,Tropical Storm Cristina,40.0
1,Wildfires,Wildfire in Australia 1028883,16348.0


#### Najdłużej trwające zdarzenie  w  danej kategorii
```sql
WITH MaksymalneCzasy AS (
    SELECT 
        c.id AS category_id,
        MAX(e.end_date - e.start_date) AS max_how_long
    FROM category AS c
    JOIN event_category AS ec ON ec.category_id = c.id
    JOIN event AS e ON e.id = ec.event_id
    WHERE e.end_date IS NOT NULL
    GROUP BY c.id
)
SELECT 
    c.title AS category_name,
    e.title AS event_name,
    (e.end_date - e.start_date) AS how_long
FROM category AS c
JOIN event_category AS ec ON ec.category_id = c.id
JOIN event AS e ON e.id = ec.event_id
JOIN MaksymalneCzasy mc 
    ON mc.category_id = c.id 
    AND (e.end_date - e.start_date) = mc.max_how_long
```

In [10]:
from scripts import analyze_5 as max_time_from_category
with engine.begin() as  conn:
    res = pd.read_sql(max_time_from_category, con= conn)
    display(res)

,category_name,event_name,how_long
0,Floods,Flood in Myanmar 1103937,2 days 04:00:00
1,Wildfires,Wildfire in The Democratic Republic of Congo 1...,7 days 05:00:00


#### Robudowany widok eventu
```sql
DROP VIEW IF EXISTS event_summary;
CREATE VIEW event_summary AS
SELECT 
    e.id as event_id,
    e.title as event_title,
    e.start_date as start_date,
    e.end_date as end_date,
    el.country_code as country_code,
    el.country as country,
    el.state as state,
    el.city as city,
    g.magnitude as magnitude,
    g.unit as unit,
    g.measure_date as last_measure_date
FROM event_location AS el
JOIN event AS e ON e.id = el.event_id
JOIN event_geometry AS eg ON eg.event_id = e.id
JOIN geometry AS g ON g.id = eg.geometry_id
WHERE g.id = (SELECT eg_sub.geometry_id 
    FROM event_geometry AS eg_sub
    JOIN geometry AS g_sub ON eg_sub.geometry_id = g_sub.id
    WHERE eg_sub.event_id = e.id 
    ORDER BY g_sub.measure_date DESC
    LIMIT 1);
```

In [11]:
from scripts import event_summary
with engine.begin() as  conn:
    conn.execute(text(event_summary))
    res = pd.read_sql("SELECT * FROM event_summary LIMIT 3", con = conn)
    display(res)

,event_id,event_title,start_date,end_date,country_code,country,state,city,magnitude,unit,last_measure_date
0,20292,Flood in Türkiye 1103920,2026-06-27 20:00:00,2026-06-29,TR,Turkey,Osmaniye,Düziçi,NaN,NaN,2026-06-27 20:00:00
1,20484,Wildfire in Zambia 1028904,2026-06-12 19:00:00,2026-06-13,ZM,Zambia,Southern Province,Itezhi-Tezhi,7491.0,hectare,2026-06-12 19:00:00
2,20450,Wildfire in Australia 1028879,2026-06-11 19:00:00,2026-06-11,AU,Australia,Queensland,Mission River,5273.0,hectare,2026-06-11 19:00:00


## Analiza Danych i Wizualizacja